# Personal Finance & Spending Analysis
## 01 — Data Exploration

**Project goal:** Explore personal income, household expenses, and savings-related data before cleaning and deeper analysis.

**Notebook workflow**
1. Import libraries and configure display settings
2. Load the raw dataset
3. Inspect dimensions, columns, and data types
4. Check missing values and duplicate records
5. Review numerical and categorical variables
6. Explore distributions and relationships
7. Record initial observations

> **Before running:** Place your dataset at `data/raw_data.csv` in the project root. This notebook is saved inside the `notebooks/` folder, so the default relative path is `../data/raw_data.csv`. You can change `DATA_PATH` in the next section if your file is somewhere else.

**Important:** This notebook explores the data. It does not overwrite or clean the original CSV file.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Visualization settings
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)

print("Libraries imported successfully!")

: 

## 2. Load Dataset
Update `DATA_PATH` if your CSV file is stored in a different location.

In [ ]:
DATA_PATH = Path("../data/raw_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Put raw_data.csv in the project's data/ folder, "
        "or update DATA_PATH to the correct CSV location."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"File: {DATA_PATH.resolve()}")

## 3. Initial Data Inspection

In [ ]:
# Preview the first 5 records
display(df.head())

In [ ]:
# Preview the last 5 records
display(df.tail())

In [ ]:
# Random sample of 5 records
display(df.sample(min(5, len(df)), random_state=42))

In [ ]:
# Dataset dimensions
rows, columns = df.shape

print(f"Total records (rows): {rows:,}")
print(f"Total features (columns): {columns:,}")

In [ ]:
# Column names
print("Column names:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

In [ ]:
# Data types and non-null counts
df.info()

## 4. Data Quality Checks

### 4.1 Missing values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean().mul(100)
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary.style.format({"missing_percentage": "{:.2f}%"}))

### 4.2 Duplicate records

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df) * 100) if len(df) else 0

print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

### 4.3 Data types

In [ ]:
dtype_summary = pd.DataFrame({
    "column": df.columns,
    "data_type": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values
})
display(dtype_summary)

## 5. Statistical Summary

In [ ]:
# Summary statistics for numerical columns
display(df.describe().T)

In [ ]:
# Summary statistics for all columns, including categorical columns
display(df.describe(include="all").T)

## 6. Categorical Variable Exploration

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Categorical columns:", categorical_columns if categorical_columns else "None detected")

for column in categorical_columns:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).rename("count").to_frame())

## 7. Numerical Variable Distributions

In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()

print(f"Numerical columns detected: {len(numerical_columns)}")

if numerical_columns:
    df[numerical_columns].hist(
        bins=25,
        figsize=(16, max(6, 3 * int(np.ceil(len(numerical_columns) / 3)))),
        edgecolor="black"
    )
    plt.suptitle("Distributions of Numerical Variables", y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("No numerical columns available for plotting.")

## 8. Potential Outlier Review
Boxplots are used here to visually flag unusually high or low values. An outlier is not automatically an error; it should be checked against business context.

In [ ]:
# Select up to 8 numerical columns per figure to keep charts readable
for start in range(0, len(numerical_columns), 8):
    selected_columns = numerical_columns[start:start + 8]
    if not selected_columns:
        continue

    plot_data = df[selected_columns].melt(var_name="variable", value_name="value")
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=plot_data, x="variable", y="value")
    plt.title("Boxplots — " + ", ".join(selected_columns))
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Correlation Analysis
Correlation measures the strength and direction of linear relationships between numerical variables. Correlation does not prove causation.

In [ ]:
if len(numerical_columns) >= 2:
    correlation_matrix = df[numerical_columns].corr()

    plt.figure(figsize=(16, 12))
    sns.heatmap(
        correlation_matrix,
        cmap="coolwarm",
        center=0,
        annot=False,
        linewidths=0.3
    )
    plt.title("Correlation Heatmap of Numerical Variables")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numerical columns are required for correlation analysis.")

## 10. Initial Finance-Focused Checks
These checks run only when the relevant columns exist in the dataset.

In [ ]:
# Inspect key financial columns when available
finance_columns = [
    "Income", "Disposable_Income", "Desired_Savings",
    "Desired_Savings_Percentage", "Rent", "Groceries",
    "Transport", "Eating_Out", "Entertainment", "Utilities",
    "Healthcare", "Education", "Miscellaneous", "Loan_Repayment",
    "Insurance"
]

available_finance_columns = [col for col in finance_columns if col in df.columns]
missing_finance_columns = [col for col in finance_columns if col not in df.columns]

print("Available finance columns:")
print(available_finance_columns)

if missing_finance_columns:
    print("\nNot present in this dataset:")
    print(missing_finance_columns)

if available_finance_columns:
    display(df[available_finance_columns].describe().T)

In [ ]:
# Average income by occupation, if both columns exist
if {"Occupation", "Income"}.issubset(df.columns):
    income_by_occupation = (
        df.groupby("Occupation", dropna=False)["Income"]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("mean", ascending=False)
    )
    display(income_by_occupation)
else:
    print("Occupation and/or Income column not found; skipped this analysis.")

In [ ]:
# Average income by city tier, if both columns exist
if {"City_Tier", "Income"}.issubset(df.columns):
    income_by_city_tier = (
        df.groupby("City_Tier", dropna=False)["Income"]
        .agg(["count", "mean", "median"])
        .sort_values("mean", ascending=False)
    )
    display(income_by_city_tier)
else:
    print("City_Tier and/or Income column not found; skipped this analysis.")